# ДЗ. Пороговая обработка изображений

Старинская Светлана, 3ФИИТ

### 1. Импорт библиотек и константы

Работа выполнена по материалам практики 02. Для локального запуска при отсутствии OpenCV можно установить `opencv-python-headless`.

In [ ]:
# Для локального Jupyter при необходимости:
# !pip install opencv-python-headless

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
ADAPTIVE_BLOCK_SIZE = 31
ADAPTIVE_C = 5
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (12, 6)

### 2. Подготовка изображения и базовая функция

Создаём изображение из практики: на градиентном фоне есть светлые объекты и шум.

In [ ]:
def create_threshold_demo_image():
    h, w = 240, 360
    background = np.tile(np.linspace(60, 180, w, dtype=np.uint8), (h, 1))
    image = background.copy()

    cv2.rectangle(image, (30, 40), (130, 180), 230, -1)
    cv2.circle(image, (240, 120), 45, 250, -1)
    cv2.putText(image, 'CV', (140, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.5, 220, 3)

    noise = np.random.normal(0, 12, (h, w)).astype(np.int16)
    return np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

def demo_simple_threshold(gray_img, thresh_value):
    _, binary = cv2.threshold(gray_img, thresh_value, 255, cv2.THRESH_BINARY)
    return binary

img_demo = create_threshold_demo_image()
plt.imshow(img_demo, cmap='gray')
plt.title('Исходное изображение с неоднородным фоном')
plt.axis('off')
plt.show()

### 3. Задание 1. Сравнение глобального порога

Проверяем значения 80, 120, 160 и 200.

In [ ]:
thresholds = [80, 120, 160, 200]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(img_demo, cmap='gray')
axes[0].set_title('Исходное')
axes[0].axis('off')

for axis, threshold in zip(axes[1:], thresholds):
    binary = demo_simple_threshold(img_demo, threshold)
    axis.imshow(binary, cmap='gray')
    axis.set_title(f'Порог {threshold}')
    axis.axis('off')

plt.tight_layout()
plt.show()

**Вывод.** При 80 и 120 в белую область попадает значительная часть светлого фона. При 160 фон ещё заметен справа. Порог **200** лучше всего отделяет прямоугольник, круг и текст от фона, хотя отдельные шумовые точки остаются.

### 4. Задание 2. Otsu и адаптивная бинаризация

Сравниваем автоматический глобальный порог Оцу, Adaptive Mean и Adaptive Gaussian.

In [ ]:
otsu_value, binary_otsu = cv2.threshold(
    img_demo, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)
binary_mean = cv2.adaptiveThreshold(
    img_demo, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY,
    ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C
)
binary_gaussian = cv2.adaptiveThreshold(
    img_demo, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
    ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C
)

images = [img_demo, binary_otsu, binary_mean, binary_gaussian]
titles = ['Исходное', f'Otsu: {otsu_value:.0f}', 'Adaptive Mean', 'Adaptive Gaussian']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, image, title in zip(axes, images, titles):
    axis.imshow(image, cmap='gray')
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

**Вывод.** Otsu выбирает один порог для всего изображения, поэтому на градиентном фоне часть правой области смешивается с объектами. Адаптивные методы рассчитывают порог локально: объекты сохраняются лучше. Mean сильнее реагирует на локальные изменения, а Gaussian использует взвешенное окружение и обычно даёт более ровный результат около границ.

### 5. Задание 3. Влияние сглаживания перед Otsu

Добавляем `cv2.GaussianBlur(gray_img, (5, 5), 0)` и сравниваем результат с бинаризацией без сглаживания.

In [ ]:
blurred_demo = cv2.GaussianBlur(img_demo, (5, 5), 0)
raw_otsu_value, raw_otsu = cv2.threshold(
    img_demo, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)
blur_otsu_value, blur_otsu = cv2.threshold(
    blurred_demo, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, image, title in zip(
    axes,
    [img_demo, raw_otsu, blurred_demo, blur_otsu],
    ['Исходное', f'Otsu без blur: {raw_otsu_value:.0f}', 'GaussianBlur (5×5)', f'Otsu после blur: {blur_otsu_value:.0f}']
):
    axis.imshow(image, cmap='gray')
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.show()

**Вывод.** GaussianBlur подавляет мелкие шумовые точки, поэтому после Otsu области объектов выглядят более цельными. Слишком сильное размытие ухудшило бы тонкие границы и буквы, но ядро 5×5 сохраняет их достаточно хорошо.

### 6. Задание 4. Реальная фотография и сохранение результатов

Используется изображение с текстом и тенями на фоне. Файл лежит в `assets/text_on_uneven_background.jpg`; результаты будут сохранены в `results_task_4`.

In [ ]:
SOURCE_IMAGE = Path('assets/text_on_uneven_background.jpg')
RESULTS_DIR = Path('results_task_4')
RESULTS_DIR.mkdir(exist_ok=True)

image_bgr = cv2.imread(str(SOURCE_IMAGE))
if image_bgr is None:
    raise FileNotFoundError(f'Не найден файл: {SOURCE_IMAGE}')

gray_img = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
_, global_binary = cv2.threshold(gray_img, 150, 255, cv2.THRESH_BINARY)
otsu_real_value, otsu_binary = cv2.threshold(
    gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)
adaptive_binary = cv2.adaptiveThreshold(
    gray_img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 5
)

cv2.imwrite(str(RESULTS_DIR / '01_gray.png'), gray_img)
cv2.imwrite(str(RESULTS_DIR / '02_global.png'), global_binary)
cv2.imwrite(str(RESULTS_DIR / '03_otsu.png'), otsu_binary)
cv2.imwrite(str(RESULTS_DIR / '04_adaptive_gaussian.png'), adaptive_binary)

images = [cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), gray_img, global_binary, otsu_binary, adaptive_binary]
titles = ['Исходное', 'Gray', 'Глобальный: 150', f'Otsu: {otsu_real_value:.0f}', 'Adaptive Gaussian']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for axis, image, title in zip(axes, images, titles):
    axis.imshow(image, cmap='gray' if image.ndim == 2 else None)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'comparison.png', dpi=160, bbox_inches='tight')
plt.show()

print('Сохранено:')
for path in sorted(RESULTS_DIR.iterdir()):
    print('-', path)

**Вывод.** На реальной фотографии глобальный порог сильнее зависит от освещения листа. Otsu подбирает порог автоматически, но также использует одно значение для всего кадра. Adaptive Gaussian лучше учитывает локальную яркость, поэтому подходит для изображения с тенями и неоднородным фоном.